# TC5061 | Actividad: Dataset - Notebook | Baseline

**Alumna:** Kimberly Ruiz (A01840607)  
**Curso:** Operaciones de aprendizaje automático (Gpo 10)  
**Fecha:** 4 de octubre de 2026

## Problema

Un prestamista quiere estimar la probabilidad de que un crédito personal caiga en default antes de aprobarlo. Es un problema de **clasificación binaria**: `loan_status = 1` es default y `loan_status = 0` es un crédito que se paga.

El objetivo no es el mejor modelo. Es recorrer cada etapa del flujo, documentar las decisiones y dejar un punto de partida reproducible en MLflow.

**Fuente:** Credit Risk Dataset, Kaggle (usuario laotse), licencia CC0. https://www.kaggle.com/datasets/laotse/credit-risk-dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
DATA_PATH = Path("../data/credit_risk_dataset.csv")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## Paso 1. Carga y descripción del dataset

In [ ]:
# Carga del dataset y vista general
df_raw = pd.read_csv(DATA_PATH)
print(f"Registros: {df_raw.shape[0]:,} | Variables: {df_raw.shape[1]}")
df_raw.head()

In [ ]:
# Tipo de dato, nulos y valores únicos por variable
resumen = pd.DataFrame({
    "tipo": df_raw.dtypes.astype(str),
    "nulos": df_raw.isna().sum(),
    "pct_nulos": (df_raw.isna().mean() * 100).round(2),
    "valores_unicos": df_raw.nunique(),
})
resumen

### Diccionario de datos

El dataset tiene **32,581 registros y 12 variables**: 8 numéricas y 4 categóricas. Cada registro es un crédito.

| Variable | Tipo | Significado |
|---|---|---|
| `person_age` | Numérica | Edad del solicitante |
| `person_income` | Numérica | Ingreso anual |
| `person_home_ownership` | Categórica | Vivienda: RENT, MORTGAGE, OWN, OTHER |
| `person_emp_length` | Numérica | Antigüedad laboral en años |
| `loan_intent` | Categórica | Propósito del crédito (educación, médico, consolidación, etc.) |
| `loan_grade` | Categórica | Calificación de riesgo asignada por el prestamista, de A (mejor) a G |
| `loan_amnt` | Numérica | Monto del crédito |
| `loan_int_rate` | Numérica | Tasa de interés (%) |
| `loan_status` | **Objetivo** | 1 = default, 0 = pagado |
| `loan_percent_income` | Numérica | Monto del crédito como proporción del ingreso |
| `cb_person_default_on_file` | Categórica | Default previo en buró (Y/N) |
| `cb_person_cred_hist_length` | Numérica | Años de historial crediticio |

**Notas sobre la fuente:**
- Kaggle describe los datos como *simulados* a partir de información de buró. Sirven para practicar el flujo, no para sacar conclusiones sobre una población real.
- Solo hay dos variables con nulos: `loan_int_rate` (9.56%) y `person_emp_length` (2.75%).
- No hay identificador de cliente ni fecha. Esto limita la detección de duplicados y la validación temporal.

## Paso 2. Análisis exploratorio (EDA)

In [ ]:
# Estadísticas descriptivas de variables numéricas
df_raw.describe().T

In [ ]:
# Estadísticas de variables categóricas
df_raw.describe(include="object").T

In [ ]:
# Distribución de la variable objetivo: ¿qué tan desbalanceado está el problema?
conteo = df_raw["loan_status"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(["No default (0)", "Default (1)"], conteo.values, color=["#4C72B0", "#C44E52"])
for i, v in enumerate(conteo.values):
    ax.text(i, v, f"{v:,}\n({v / conteo.sum():.1%})", ha="center", va="bottom")
ax.set_title("Distribución de loan_status")
ax.set_ylabel("Créditos")
ax.set_ylim(0, conteo.max() * 1.2)
plt.tight_layout()
plt.show()

In [ ]:
# Histogramas de las variables numéricas para ver forma, sesgo y colas
num_cols = ["person_age", "person_income", "person_emp_length", "loan_amnt",
            "loan_int_rate", "loan_percent_income", "cb_person_cred_hist_length"]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flat, num_cols):
    ax.hist(df_raw[col].dropna(), bins=40, color="#4C72B0", edgecolor="white")
    ax.set_title(col)
axes.flat[-1].axis("off")
plt.suptitle("Distribución de variables numéricas", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots para detectar outliers extremos
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["person_age", "person_emp_length", "person_income"]):
    ax.boxplot(df_raw[col].dropna(), vert=False)
    ax.set_title(col)
plt.suptitle("Outliers en edad, antigüedad laboral e ingreso", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Tasa de default por variable categórica: ¿qué segmentos son más riesgosos?
cat_cols = ["loan_grade", "person_home_ownership", "loan_intent", "cb_person_default_on_file"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, cat_cols):
    tasa = df_raw.groupby(col)["loan_status"].mean().sort_values()
    ax.barh(tasa.index, tasa.values, color="#C44E52")
    ax.axvline(df_raw["loan_status"].mean(), color="black", ls="--", lw=1, label="Promedio")
    ax.set_title(f"Tasa de default por {col}")
    ax.set_xlim(0, 1)
axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlación entre numéricas y la variable objetivo
corr = df_raw[num_cols + ["loan_status"]].corr()
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im)
ax.set_title("Correlación de Pearson")
plt.tight_layout()
plt.show()

In [ ]:
# La tasa de interés la asigna el prestamista según el grado: lo confirmamos
df_raw.groupby("loan_grade")["loan_int_rate"].agg(["median", "count"])

### Hallazgos del EDA

1. **Clases desbalanceadas.** 78.2% de los créditos se pagan y 21.8% caen en default. Un modelo que siempre predice "paga" obtiene 78% de accuracy sin aprender nada. Por eso accuracy no sirve como métrica principal.
2. **Outliers imposibles.** `person_age` llega a 144 años y `person_emp_length` a 123 años. Son errores de captura, no clientes reales.
3. **Ingreso con cola larga.** La mediana es 55,000 y el máximo 6,000,000. La distribución está muy sesgada a la derecha. Un modelo lineal se distorsiona con esa escala.
4. **El grado de riesgo separa muy bien.** La tasa de default sube de 10% en grado A a 98% en grado G.
5. **La tasa de interés depende del grado.** La mediana sube de 7.49% en A a 20.16% en G. Esto importa para imputar: la tasa no falta al azar respecto al grado.
6. **Vivienda y default previo en buró importan.** Quien renta cae en default 31.6% de las veces; quien es dueño, 7.5%. Con default previo en buró la tasa es 37.8%, contra 18.4% sin él.
7. **Correlaciones con el objetivo.** Las más altas son `loan_percent_income` (0.38) y `loan_int_rate` (0.34). El ingreso tiene correlación negativa (-0.14). La edad casi no se relaciona con el default (-0.02).

**Implicación:** `loan_grade` y `loan_int_rate` son decisiones del propio prestamista. Reflejan un modelo de riesgo anterior. El baseline las usa, pero un modelo en producción debe revisar si las conoce en el momento de la solicitud.